# Standalone Foundation OOF Reproduction

This notebook creates the foundation out-of-fold (OOF) predictions directly from train.csv and test.csv, then verifies the historical result with exact metric equality. It does not import project code or read prior model outputs.

Run it in a clean CUDA environment with Python 3.13.15, the package versions listed below, and at least 10 GiB free GPU memory. The historical reference used an NVIDIA GeForce RTX 3060 with 12 GB VRAM, reporting 11.63 GiB total and 11.52 GiB free before training. Expect about one hour on comparable hardware.

## Steps

1. Configure the data and output folders, then check the package versions, GPU memory, and data fingerprints.
2. Prepare the recorded features and define the CatBoost, TabM, and TabICL models.
3. Train all three models with three-fold cross-validation and create the OOF predictions.
4. Blend the source models with TabICL, write the OOF and test-prediction files, and compare the result with the historical metrics.

In [ ]:
import copy
from dataclasses import dataclass
import gc
import hashlib
import importlib.metadata
import json
from pathlib import Path
import platform
import time

import numpy as np
import pandas as pd
from IPython.display import display
from catboost import CatBoostClassifier
from rtdl_num_embeddings import PiecewiseLinearEmbeddings, compute_bins
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import QuantileTransformer
from tabicl import TabICLClassifier
from tabm import TabM
import torch
from torch.nn import functional as F


In [ ]:
DATA_DIRECTORY = None
RUN_DIRECTORY = None
RUN_NAME = "foundation-reproduction-standalone"

ID_COLUMN = "claim_id"
TARGET_COLUMN = "label"
CATEGORICAL_COLUMNS = [
    "kdkc", "dati2", "typeppk", "jkpst", "jnspelsep", "cmg", "severitylevel", "diagprimer",
]
EXPECTED_DATA_SHA256 = {
    "train.csv": "e87c42e644e07745e1fdf270949a438e4d636e1497bbcc48b1bea032279ddc3b",
    "test.csv": "4df0cfa840ecf44bc2c6aa09270261f63e939341d276c7b75df1bde4f636b36e",
}
EXPECTED_DATA_ROWS = {"train": 160174, "test": 40043}
EXPECTED_VERSIONS = {
    "python": "3.13.15", "catboost": "1.2.10", "tabm": "0.0.3",
    "rtdl-num-embeddings": "0.0.12", "torch": "2.13.0", "tabicl": "2.2.0",
    "numpy": "2.5.2", "pandas": "3.0.5", "scikit-learn": "1.9.0",
}
CTR_PARAMS = {
    "loss_function": "Logloss", "eval_metric": "PRAUC", "iterations": 10000,
    "learning_rate": 0.03, "depth": 8, "l2_leaf_reg": 10.0, "verbose": False,
    "allow_writing_files": False, "random_strength": 2.0, "bagging_temperature": 1.0,
    "max_ctr_complexity": 4, "task_type": "GPU", "devices": "0",
}
TABM_PARAMS = {
    "k": 16, "d_block": 352, "n_blocks": 4, "dropout": 0.17446594614882407,
    "learning_rate": 0.0022427972615287574, "weight_decay": 0.006004195999587353,
    "batch_size": 512, "max_epochs": 500, "patience": 20,
    "inner_validation_fraction": 0.1, "piecewise_bins": 79, "piecewise_embedding_dim": 25,
}
FOUNDATION_PARAMS = {
    "n_estimators": 4, "estimator_batch_size": 1, "prediction_chunk_size": 1024,
    "min_free_vram_gib": 10.0, "tabicl_cache_mode": "repr",
}
FOLD_COUNT = 3
FOLD_RANDOM_STATE = 42
SOURCE_SEEDS = (42, 2026)
FOUNDATION_SEED = 42
SELECTED_FOUNDATION_WEIGHT = 0.5
CANDIDATE_NAME = "foundation_ctr_tabm_base_blend_w50"

if DATA_DIRECTORY is None:
    candidates = [Path.cwd() / "data", Path.cwd().parent / "data"]
    DATA_DIRECTORY = next((path for path in candidates if (path / "train.csv").exists()), None)
else:
    DATA_DIRECTORY = Path(DATA_DIRECTORY)
if DATA_DIRECTORY is None:
    raise FileNotFoundError("Set DATA_DIRECTORY to the folder containing train.csv and test.csv.")
RUN_DIRECTORY = Path.cwd() / RUN_NAME if RUN_DIRECTORY is None else Path(RUN_DIRECTORY)
if RUN_DIRECTORY.exists() and any(RUN_DIRECTORY.iterdir()):
    raise FileExistsError(f"Run directory already contains artifacts: {RUN_DIRECTORY}")

EXPECTED_METRICS = {
    "average_precision": 0.8311228283421712, "brier_score": 0.169094052054208,
    "fraud_caught_at_3pct": 4743, "fraud_caught_at_5pct": 7845, "fraud_caught_at_7pct": 10857,
    "fraud_prevalence": 0.5007242124189943, "legitimate_audits_at_3pct": 62,
    "legitimate_audits_at_5pct": 163, "legitimate_audits_at_7pct": 355,
    "lift_at_3pct": 1.9713382131550867, "lift_at_5pct": 1.9564569284810422,
    "lift_at_7pct": 1.9338739200616288, "log_loss": 0.5069609155303831,
    "mean_prediction": 0.5015233781612578, "n_audited_at_3pct": 4805,
    "n_audited_at_5pct": 8008, "n_audited_at_7pct": 11212, "n_rows": 160174,
    "normalized_recall_at_3pct": 0.9870967741935484,
    "normalized_recall_at_5pct": 0.9796453546453546,
    "normalized_recall_at_7pct": 0.9683374955404923,
    "precision_at_3pct": 0.9870967741935484, "precision_at_5pct": 0.9796453546453546,
    "precision_at_7pct": 0.9683374955404923, "recall_at_3pct": 0.05913743874917397,
    "recall_at_5pct": 0.09781429622333329, "recall_at_7pct": 0.13536900115955763,
    "roc_auc": 0.8283982288959596,
}


## Step 1 — Validate the environment and data

The data fingerprints and library versions are strict guards. A mismatch stops the replay before training begins, preventing a result from a different dataset or software stack.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def validate_arrays(labels, probabilities):
    labels = np.asarray(labels, dtype=int).reshape(-1)
    probabilities = np.asarray(probabilities, dtype=float).reshape(-1)
    if len(labels) != len(probabilities) or not len(labels):
        raise ValueError("Labels and probabilities must have the same non-zero length.")
    if not np.isin(labels, [0, 1]).all():
        raise ValueError("Labels must contain only 0 and 1.")
    if not np.isfinite(probabilities).all() or ((probabilities < 0) | (probabilities > 1)).any():
        raise ValueError("Probabilities must be finite values within [0, 1].")
    return labels, probabilities


def evaluate_probabilities(labels, probabilities):
    labels, probabilities = validate_arrays(labels, probabilities)
    total_fraud = int(labels.sum())
    metrics = {
        "n_rows": len(labels), "fraud_prevalence": float(labels.mean()),
        "mean_prediction": float(probabilities.mean()),
        "brier_score": float(brier_score_loss(labels, probabilities)),
        "average_precision": float(average_precision_score(labels, probabilities)),
        "roc_auc": float(roc_auc_score(labels, probabilities)),
        "log_loss": float(log_loss(labels, probabilities, labels=[0, 1])),
    }
    for fraction in (0.03, 0.05, 0.07):
        suffix = f"{fraction:.0%}".replace("%", "pct")
        audited_rows = int(np.floor(len(labels) * fraction))
        selected = np.argsort(-probabilities, kind="mergesort")[:audited_rows]
        caught = int(labels[selected].sum())
        precision = caught / audited_rows if audited_rows else 0.0
        maximum = min(audited_rows, total_fraud)
        metrics.update({
            f"n_audited_at_{suffix}": audited_rows,
            f"fraud_caught_at_{suffix}": caught,
            f"legitimate_audits_at_{suffix}": audited_rows - caught,
            f"recall_at_{suffix}": caught / total_fraud if total_fraud else 0.0,
            f"normalized_recall_at_{suffix}": caught / maximum if maximum else 0.0,
            f"precision_at_{suffix}": precision,
            f"lift_at_{suffix}": precision / (total_fraud / len(labels)) if total_fraud else 0.0,
        })
    return metrics


def categorical_values(values):
    return values.astype("string").fillna("__MISSING__").astype(str)


def prepare_features(train, test, extended):
    features = [column for column in train.columns if column not in {ID_COLUMN, TARGET_COLUMN}]
    if features != [column for column in test.columns if column != ID_COLUMN]:
        raise ValueError("Train and test feature schemas differ.")
    X, X_test = train.loc[:, features].copy(), test.loc[:, features].copy()
    for column in CATEGORICAL_COLUMNS:
        X[column], X_test[column] = categorical_values(X[column]), categorical_values(X_test[column])
    categorical_features = list(CATEGORICAL_COLUMNS)
    if extended:
        diagnosis_columns = [column for column in features if column.startswith("dx2_")]
        procedure_columns = [column for column in features if column.startswith("proc")]
        for frame in (X, X_test):
            frame["secondary_diagnosis_count"] = frame[diagnosis_columns].apply(pd.to_numeric, errors="coerce").sum(axis=1, min_count=1)
            frame["procedure_count"] = frame[procedure_columns].apply(pd.to_numeric, errors="coerce").sum(axis=1, min_count=1)
            frame["secondary_diagnosis_count_bucket"] = pd.cut(frame["secondary_diagnosis_count"], [-np.inf, 0, 1, 2, np.inf], labels=["0", "1", "2", "3+"], include_lowest=True).astype("string").fillna("__MISSING__").astype(str)
            frame["procedure_count_bucket"] = pd.cut(frame["procedure_count"], [-np.inf, 0, 1, 2, 3, np.inf], labels=["0", "1", "2", "3", "4+"], include_lowest=True).astype("string").fillna("__MISSING__").astype(str)
        categorical_features.extend(["secondary_diagnosis_count_bucket", "procedure_count_bucket"])
        interactions = {
            "typeppk_cmg": ("typeppk", "cmg"), "cmg_severitylevel": ("cmg", "severitylevel"),
            "diagprimer_cmg": ("diagprimer", "cmg"), "dati2_typeppk": ("dati2", "typeppk"),
        }
        for name, columns in interactions.items():
            for frame in (X, X_test):
                frame[name] = frame.loc[:, list(columns)].astype("string").fillna("__MISSING__").agg("__".join, axis=1).astype(str)
            categorical_features.append(name)
        for frame in (X, X_test):
            los = pd.to_numeric(frame["los"], errors="coerce")
            frame["los_zero_indicator"] = np.where(los.notna(), (los == 0).astype(int), np.nan)
            frame["los_bucket"] = pd.cut(los, [-np.inf, 0, 1, 3, 7, 14, np.inf], labels=["0", "1", "2-3", "4-7", "8-14", "15+"], include_lowest=True).astype("string").fillna("__MISSING__").astype(str)
        categorical_features.append("los_bucket")
    if list(X.columns) != list(X_test.columns):
        raise RuntimeError("Prepared train and test features are not aligned.")
    return X, train[TARGET_COLUMN].astype(int), X_test, categorical_features


actual_versions = {name: importlib.metadata.version(name) for name in EXPECTED_VERSIONS if name != "python"}
actual_versions["python"] = platform.python_version()
version_mismatches = {name: {"expected": version, "actual": actual_versions[name]} for name, version in EXPECTED_VERSIONS.items() if actual_versions[name] != version}
if version_mismatches:
    raise RuntimeError(f"Package version mismatch: {version_mismatches}")
if torch.version.cuda != "13.0":
    raise RuntimeError(f"Expected CUDA 13.0, found {torch.version.cuda}.")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this replay.")
free_bytes, total_bytes = torch.cuda.mem_get_info(torch.device("cuda"))
if free_bytes < int(FOUNDATION_PARAMS["min_free_vram_gib"] * 1024**3):
    raise RuntimeError(f"At least 10 GiB free VRAM is required; found {free_bytes / 1024**3:.2f} GiB.")

for filename, expected_hash in EXPECTED_DATA_SHA256.items():
    actual_hash = sha256_file(DATA_DIRECTORY / filename)
    if actual_hash != expected_hash:
        raise ValueError(f"{filename} does not match the recorded dataset fingerprint.")
dtype_map = {column: "string" for column in CATEGORICAL_COLUMNS}
train = pd.read_csv(DATA_DIRECTORY / "train.csv", dtype=dtype_map)
test = pd.read_csv(DATA_DIRECTORY / "test.csv", dtype=dtype_map)
if len(train) != EXPECTED_DATA_ROWS["train"] or len(test) != EXPECTED_DATA_ROWS["test"]:
    raise ValueError("Dataset row counts do not match the recorded experiment.")
if ID_COLUMN not in train or ID_COLUMN not in test or TARGET_COLUMN not in train or TARGET_COLUMN in test:
    raise ValueError("Data files do not have the required claim_id and label schema.")
display(pd.DataFrame([{
    "gpu": torch.cuda.get_device_name(torch.device("cuda")),
    "free_vram_gib": free_bytes / 1024**3, "total_vram_gib": total_bytes / 1024**3,
    "train_rows": len(train), "test_rows": len(test),
}]))


## Step 2 — Prepare and define the three models

For OOF prediction, the data is split into three folds. For each fold, a model trains on the other two folds and predicts only the held-out fold. After three rounds, every training row has one prediction from a model that did not see that row during training.

1. CatBoost and TabM each run twice, using seeds 42 and 2026; their predictions are averaged to reduce seed variation.
2. TabICL runs once with seed 42 using the base feature set.
3. The final candidate blends 50% TabICL with a base blend of 50% CatBoost and 50% TabM.

In [ ]:
def train_ctr_cv(X, y, X_test, categorical_features, cv, seed):
    oof = np.full(len(X), np.nan, dtype=float)
    test_predictions, folds = [], np.full(len(X), -1, dtype=int)
    params = {**CTR_PARAMS, "random_seed": seed}
    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):
        model = CatBoostClassifier(**params)
        model.fit(X.iloc[train_idx], y.iloc[train_idx], cat_features=categorical_features, eval_set=(X.iloc[valid_idx], y.iloc[valid_idx]), early_stopping_rounds=200, verbose=False)
        oof[valid_idx] = model.predict_proba(X.iloc[valid_idx])[:, 1]
        test_predictions.append(model.predict_proba(X_test)[:, 1])
        folds[valid_idx] = fold
        del model
        gc.collect()
    validate_arrays(y, oof)
    return oof, np.mean(np.vstack(test_predictions), axis=0), folds


class TabMPreprocessor:
    def __init__(self, categorical_features, random_state):
        self.categorical_features, self.random_state = categorical_features, random_state

    def fit(self, X):
        self.numeric_features = [column for column in X.columns if column not in self.categorical_features]
        self.category_maps = {}
        for feature in self.categorical_features:
            categories = sorted(categorical_values(X[feature]).unique().tolist())
            self.category_maps[feature] = {category: index for index, category in enumerate(categories, start=1)}
        numeric = self.numeric_values(X)
        self.medians = {feature: float(np.median(values[np.isfinite(values)])) if np.isfinite(values).any() else 0.0 for feature, values in zip(self.numeric_features, numeric.T, strict=True)}
        imputed = self.impute(numeric)
        missing = ~np.isfinite(numeric)
        self.quantile_indices = [index for index in range(imputed.shape[1]) if np.unique(imputed[:, index]).size > 1]
        self.missing_indices = [index for index in range(missing.shape[1]) if np.unique(missing[:, index]).size > 1]
        self.quantile = QuantileTransformer(n_quantiles=min(1000, len(imputed)), output_distribution="normal", random_state=self.random_state).fit(imputed[:, self.quantile_indices]) if self.quantile_indices else None
        return self

    def numeric_values(self, X):
        return X.loc[:, self.numeric_features].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)

    def impute(self, values):
        values = values.copy()
        for index, feature in enumerate(self.numeric_features):
            values[~np.isfinite(values[:, index]), index] = self.medians[feature]
        return values

    @property
    def cat_cardinalities(self):
        return [len(self.category_maps[feature]) + 1 for feature in self.categorical_features]

    def transform(self, X):
        numeric = self.numeric_values(X)
        missing = (~np.isfinite(numeric)).astype(np.float32)
        parts = []
        if self.quantile_indices:
            parts.append(self.quantile.transform(self.impute(numeric)[:, self.quantile_indices]))
        if self.missing_indices:
            parts.append(missing[:, self.missing_indices])
        x_num = np.column_stack(parts).astype(np.float32) if parts else np.empty((len(X), 0), dtype=np.float32)
        x_cat = np.column_stack([categorical_values(X[feature]).map(self.category_maps[feature]).fillna(0).to_numpy(dtype=np.int64) for feature in self.categorical_features])
        return x_num, x_cat


def set_tabm_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.cuda.manual_seed_all(seed)


def batches(indices, batch_size):
    for start in range(0, len(indices), batch_size):
        yield indices[start:start + batch_size]


def cat_tensor(values, indices, device):
    return torch.as_tensor(values[indices], dtype=torch.long, device=device)


def make_tabm_model(x_num, cardinalities):
    bins = compute_bins(torch.as_tensor(x_num, dtype=torch.float32), n_bins=min(TABM_PARAMS["piecewise_bins"], len(x_num)))
    embeddings = PiecewiseLinearEmbeddings(bins, d_embedding=TABM_PARAMS["piecewise_embedding_dim"], activation=False, version="B")
    return TabM.make(n_num_features=x_num.shape[1], cat_cardinalities=cardinalities, d_out=1, num_embeddings=embeddings, k=TABM_PARAMS["k"], d_block=TABM_PARAMS["d_block"], n_blocks=TABM_PARAMS["n_blocks"], dropout=TABM_PARAMS["dropout"])


def tabm_probabilities(model, x_num, x_cat, device):
    predictions = []
    model.eval()
    with torch.inference_mode():
        for batch_idx in batches(np.arange(len(x_num)), TABM_PARAMS["batch_size"]):
            logits = model(torch.as_tensor(x_num[batch_idx], dtype=torch.float32, device=device), cat_tensor(x_cat, batch_idx, device)).squeeze(-1).float()
            predictions.append(torch.sigmoid(logits).mean(dim=1).cpu().numpy())
    probabilities = np.concatenate(predictions).astype(float)
    validate_arrays(np.zeros(len(probabilities), dtype=int), probabilities)
    return probabilities


def fit_tabm(model, x_num, x_cat, labels, seed, device):
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=TABM_PARAMS["inner_validation_fraction"], random_state=seed)
    inner_train, inner_valid = next(splitter.split(x_num, labels))
    optimizer = torch.optim.AdamW(model.parameters(), lr=TABM_PARAMS["learning_rate"], weight_decay=TABM_PARAMS["weight_decay"])
    scaler = torch.amp.GradScaler(device="cuda", enabled=True)
    random = np.random.default_rng(seed)
    best_state, best_score, stale_epochs = None, -np.inf, 0
    for _ in range(TABM_PARAMS["max_epochs"]):
        model.train()
        for batch_idx in batches(random.permutation(inner_train), TABM_PARAMS["batch_size"]):
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=True):
                logits = model(torch.as_tensor(x_num[batch_idx], dtype=torch.float32, device=device), cat_tensor(x_cat, batch_idx, device)).squeeze(-1)
                target = torch.as_tensor(labels[batch_idx], dtype=torch.float32, device=device).unsqueeze(1).expand_as(logits)
                loss = F.binary_cross_entropy_with_logits(logits, target)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        score = float(average_precision_score(labels[inner_valid], tabm_probabilities(model, x_num[inner_valid], x_cat[inner_valid], device)))
        if score > best_score + 1e-12:
            best_score, best_state, stale_epochs = score, copy.deepcopy(model.state_dict()), 0
        else:
            stale_epochs += 1
            if stale_epochs >= TABM_PARAMS["patience"]:
                break
    model.load_state_dict(best_state)


def train_tabm_cv(X, y, X_test, categorical_features, cv, seed):
    device, oof = torch.device("cuda"), np.full(len(X), np.nan, dtype=float)
    test_predictions, folds = [], np.full(len(X), -1, dtype=int)
    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):
        set_tabm_seed(seed + fold)
        preprocessor = TabMPreprocessor(categorical_features, seed + fold).fit(X.iloc[train_idx])
        train_num, train_cat = preprocessor.transform(X.iloc[train_idx])
        valid_num, valid_cat = preprocessor.transform(X.iloc[valid_idx])
        test_num, test_cat = preprocessor.transform(X_test)
        model = make_tabm_model(train_num, preprocessor.cat_cardinalities).to(device)
        fit_tabm(model, train_num, train_cat, y.iloc[train_idx].to_numpy(), seed + fold, device)
        oof[valid_idx] = tabm_probabilities(model, valid_num, valid_cat, device)
        test_predictions.append(tabm_probabilities(model, test_num, test_cat, device))
        folds[valid_idx] = fold
        del model
        gc.collect()
        torch.cuda.empty_cache()
    validate_arrays(y, oof)
    return oof, np.mean(np.vstack(test_predictions), axis=0), folds


In [ ]:
def make_tabicl(seed, cache_directory):
    return TabICLClassifier(
        n_estimators=FOUNDATION_PARAMS["n_estimators"], batch_size=FOUNDATION_PARAMS["estimator_batch_size"],
        kv_cache="repr", device="cuda", use_amp="auto", offload_mode="auto",
        disk_offload_dir=str(cache_directory) if cache_directory is not None else None, random_state=seed, verbose=False,
    )


def tabicl_probabilities(model, X):
    predictions = []
    for start in range(0, len(X), FOUNDATION_PARAMS["prediction_chunk_size"]):
        output = model.predict_proba(X.iloc[start:start + FOUNDATION_PARAMS["prediction_chunk_size"]])
        values = output.to_numpy() if isinstance(output, pd.DataFrame) else np.asarray(output)
        predictions.append(np.asarray(values[:, 1], dtype=float))
    probabilities = np.concatenate(predictions)
    validate_arrays(np.zeros(len(probabilities), dtype=int), probabilities)
    return probabilities


def preflight_tabicl(X, y, cv):
    train_idx, valid_idx = next(iter(cv.split(X, y)))
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    started = time.monotonic()
    model = make_tabicl(FOUNDATION_SEED, None)
    model.fit(X.iloc[train_idx], y.iloc[train_idx].to_numpy())
    tabicl_probabilities(model, X.iloc[valid_idx[:FOUNDATION_PARAMS["prediction_chunk_size"]]])
    result = {
        "elapsed_seconds": time.monotonic() - started,
        "peak_allocated_bytes": int(torch.cuda.max_memory_allocated()),
        "train_rows": len(train_idx), "prediction_rows": min(len(valid_idx), FOUNDATION_PARAMS["prediction_chunk_size"]),
    }
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return result


def train_tabicl_cv(X, y, X_test, cv):
    oof, folds = np.full(len(X), np.nan, dtype=float), np.full(len(X), -1, dtype=int)
    test_predictions = []
    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):
        cache_directory = RUN_DIRECTORY / "cache" / f"seed_{FOUNDATION_SEED}" / f"fold_{fold}"
        cache_directory.mkdir(parents=True, exist_ok=True)
        model = make_tabicl(FOUNDATION_SEED + fold, cache_directory)
        model.fit(X.iloc[train_idx], y.iloc[train_idx].to_numpy())
        oof[valid_idx] = tabicl_probabilities(model, X.iloc[valid_idx])
        test_predictions.append(tabicl_probabilities(model, X_test))
        folds[valid_idx] = fold
        del model
        gc.collect()
        torch.cuda.empty_cache()
    validate_arrays(y, oof)
    return oof, np.mean(np.vstack(test_predictions), axis=0), folds


## Step 3 — Create the OOF predictions

This is the long-running cell. It trains each fold, combines the held-out predictions into one OOF file, then applies the recorded 50% foundation blend. It also averages fold predictions for the test set and writes both outputs under RUN_DIRECTORY.

In [ ]:
RUN_DIRECTORY.mkdir(parents=True, exist_ok=True)
started = time.monotonic()
cv = StratifiedKFold(n_splits=FOLD_COUNT, shuffle=True, random_state=FOLD_RANDOM_STATE)
ctr_X, y, ctr_test_X, ctr_categorical = prepare_features(train, test, extended=True)
foundation_X, foundation_y, foundation_test_X, _ = prepare_features(train, test, extended=False)

ctr_runs = [train_ctr_cv(ctr_X, y, ctr_test_X, ctr_categorical, cv, seed) for seed in SOURCE_SEEDS]
ctr_oof = np.mean(np.vstack([result[0] for result in ctr_runs]), axis=0)
ctr_test = np.mean(np.vstack([result[1] for result in ctr_runs]), axis=0)
tabm_runs = [train_tabm_cv(ctr_X, y, ctr_test_X, ctr_categorical, cv, seed) for seed in SOURCE_SEEDS]
tabm_oof = np.mean(np.vstack([result[0] for result in tabm_runs]), axis=0)
tabm_test = np.mean(np.vstack([result[1] for result in tabm_runs]), axis=0)
expected_folds = ctr_runs[0][2]
if not all(np.array_equal(result[2], expected_folds) for result in [*ctr_runs, *tabm_runs]):
    raise RuntimeError("Source models produced inconsistent fold assignments.")

preflight = preflight_tabicl(foundation_X, foundation_y, cv)
foundation_oof, foundation_test, foundation_folds = train_tabicl_cv(foundation_X, foundation_y, foundation_test_X, cv)
if not np.array_equal(expected_folds, foundation_folds):
    raise RuntimeError("Foundation model produced inconsistent fold assignments.")

base_oof = 0.5 * ctr_oof + 0.5 * tabm_oof
base_test = 0.5 * ctr_test + 0.5 * tabm_test
screen_rows = []
for weight in (0.10, 0.25, 0.50, 0.75, 1.00):
    metrics = evaluate_probabilities(y, (1 - weight) * base_oof + weight * foundation_oof)
    screen_rows.append({"foundation_weight": weight, **metrics})
selected_oof = (1 - SELECTED_FOUNDATION_WEIGHT) * base_oof + SELECTED_FOUNDATION_WEIGHT * foundation_oof
selected_test = (1 - SELECTED_FOUNDATION_WEIGHT) * base_test + SELECTED_FOUNDATION_WEIGHT * foundation_test
validate_arrays(y, selected_oof)

oof = pd.DataFrame({
    ID_COLUMN: train[ID_COLUMN], TARGET_COLUMN: y, "fold": expected_folds,
    "fraud_probability_raw": selected_oof, "fraud_probability_final": selected_oof,
})
oof.to_csv(RUN_DIRECTORY / f"{CANDIDATE_NAME}_oof.csv", index=False)
pd.DataFrame({ID_COLUMN: test[ID_COLUMN], "fraud_probability_raw": selected_test}).to_csv(RUN_DIRECTORY / f"{CANDIDATE_NAME}_test_raw.csv", index=False)
pd.DataFrame(screen_rows).to_csv(RUN_DIRECTORY / "foundation_screen_metrics.csv", index=False)
with (RUN_DIRECTORY / "reproduction_environment.json").open("w") as handle:
    json.dump({
        "versions": actual_versions,
        "gpu": torch.cuda.get_device_name(torch.device("cuda")),
        "free_vram_gib_before_training": free_bytes / 1024**3,
        "total_vram_gib": total_bytes / 1024**3,
        "preflight": preflight,
        "runtime_seconds": time.monotonic() - started,
    }, handle, indent=2, sort_keys=True)

print(f"Created: {RUN_DIRECTORY / f'{CANDIDATE_NAME}_oof.csv'}")
display(pd.DataFrame(screen_rows)[["foundation_weight", "average_precision", "brier_score", "recall_at_5pct"]])

## Step 4 — Verify the recreated result

The final check recomputes average precision, Brier score, audit capture, recall, precision, and lift from the newly created OOF file. PASS requires exact equality for every recorded raw-OOF metric. The CSV and JSON evidence remain in RUN_DIRECTORY whether the result passes or fails.

In [ ]:
actual_metrics = evaluate_probabilities(y, selected_oof)
comparison = pd.DataFrame([
    {"metric": name, "expected": expected, "actual": actual_metrics.get(name), "exact_match": actual_metrics.get(name) == expected}
    for name, expected in sorted(EXPECTED_METRICS.items())
])
metrics_match = bool(comparison["exact_match"].all())
structure_checks = {
    "row_count": len(oof) == EXPECTED_DATA_ROWS["train"],
    "unique_claim_ids": oof[ID_COLUMN].is_unique,
    "folds": set(oof["fold"].unique()) == {0, 1, 2},
    "candidate_weight": SELECTED_FOUNDATION_WEIGHT == 0.5,
}
passed = metrics_match and all(structure_checks.values())
verdict = {
    "status": "PASS" if passed else "FAIL", "candidate": CANDIDATE_NAME,
    "metrics_exact_match": metrics_match, "structure_checks": structure_checks,
    "oof_path": str((RUN_DIRECTORY / f"{CANDIDATE_NAME}_oof.csv").resolve()),
}
comparison.to_csv(RUN_DIRECTORY / "reproduction_metric_comparison.csv", index=False)
with (RUN_DIRECTORY / "reproduction_verdict.json").open("w") as handle:
    json.dump(verdict, handle, indent=2, sort_keys=True)
display(comparison)
display(pd.DataFrame([structure_checks]))
print(json.dumps(verdict, indent=2))
if not passed:
    raise AssertionError("Foundation reproduction failed. See reproduction_verdict.json for details.")